# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:
# 1. Corpus, preprocesamiento y estadísticas DF/IDF
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import gutenberg
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from IPython.display import display

pd.set_option("display.max_colwidth", 90)

# El corpus Gutenberg de NLTK
try:
    ids_gutenberg = sorted(gutenberg.fileids())
except LookupError:
    print("Descargando corpus Gutenberg de NLTK (solo la primera vez)...")
    if not nltk.download("gutenberg", quiet=True):
        raise RuntimeError(
            "No fue posible descargar Gutenberg. Conéctate a Internet y ejecuta "
            "nltk.download('gutenberg') antes de reintentar."
        )
    ids_gutenberg = sorted(gutenberg.fileids())


titulos = {
    "austen-emma.txt": "Emma",
    "austen-persuasion.txt": "Persuasion",
    "austen-sense.txt": "Sense and Sensibility",
    "bible-kjv.txt": "King James Bible",
    "blake-poems.txt": "Poems (William Blake)",
    "bryant-stories.txt": "Stories to Tell to Children",
    "burgess-busterbrown.txt": "The Adventures of Buster Bear",
    "carroll-alice.txt": "Alice's Adventures in Wonderland",
    "chesterton-ball.txt": "The Ball and the Cross",
    "chesterton-brown.txt": "The Wisdom of Father Brown",
    "chesterton-thursday.txt": "The Man Who Was Thursday",
    "edgeworth-parents.txt": "The Parent's Assistant",
    "melville-moby_dick.txt": "Moby Dick",
    "milton-paradise.txt": "Paradise Lost",
    "shakespeare-caesar.txt": "Julius Caesar",
    "shakespeare-hamlet.txt": "Hamlet",
    "shakespeare-macbeth.txt": "Macbeth",
    "whitman-leaves.txt": "Leaves of Grass",
}

# el preprocesamiento de la práctica anterior: no elimina stopwords.
def tokenizar(texto: str) -> list[str]:
    return re.findall(r"\b\w+\b", texto.lower())

documentos = pd.DataFrame({
    "doc_id": [str(doc_id) for doc_id in ids_gutenberg],
    "titulo": [titulos.get(doc_id, doc_id.removesuffix(".txt")) for doc_id in ids_gutenberg],
    "texto": [gutenberg.raw(doc_id) for doc_id in ids_gutenberg],
})
documentos["tokens"] = documentos["texto"].map(tokenizar)
documentos["longitud"] = documentos["tokens"].map(len)
N = len(documentos)
promedio_longitud = float(documentos["longitud"].mean())

# CountVectorizer usa exactamente la misma función sobre los textos completos.
vectorizador = CountVectorizer(analyzer=tokenizar, dtype=np.float64)
X_conteos = vectorizador.fit_transform(documentos["texto"]).tocsr()  # dispersa: docs × términos
vocabulario = vectorizador.get_feature_names_out()
X_csc = X_conteos.tocsc()  # acceso eficiente a ocurrencias de un término en documentos
DF = np.diff(X_csc.indptr).astype(int)  # documentos donde aparece cada término
idf_tfidf = np.log(N / DF) + 1.0
idf_bm25 = np.log1p((N - DF + 0.5) / (DF + 0.5))

# TF-IDF de conteos reales, IDF sin smoothing y normalización L2 = coseno.
transformador_tfidf = TfidfTransformer(
    norm="l2", use_idf=True, smooth_idf=False, sublinear_tf=False
)
X_tfidf = transformador_tfidf.fit_transform(X_conteos).tocsr()  # sigue dispersa

estadisticas = pd.DataFrame({
    "termino": vocabulario,
    "df": DF,
    "idf_tfidf": idf_tfidf,
    "idf_bm25": idf_bm25,
})
indice_vocab = vectorizador.vocabulary_

assert N == 18, f"Se esperaban 18 obras de Gutenberg; aparecieron {N}."
assert np.array_equal(
    np.asarray(X_conteos.sum(axis=1)).ravel().astype(int),
    documentos["longitud"].to_numpy()
), "Los conteos no corresponden a la tokenización anterior."
assert np.allclose(transformador_tfidf.idf_, idf_tfidf)
assert X_conteos.shape == X_tfidf.shape == (N, len(vocabulario))

print(f"Corpus: {N} documentos | vocabulario: {len(vocabulario):,} términos")
print(f"Tokens totales: {documentos['longitud'].sum():,} | longitud promedio: {promedio_longitud:,.2f}")
print("\nTabla pedida (texto y tokens íntegros se conservan en 'documentos'):")
display(documentos.assign(
    texto=documentos["texto"].str.slice(0, 95) + "...",
    tokens=documentos["tokens"].map(lambda t: t[:12]),
)[["doc_id", "titulo", "texto", "tokens", "longitud"]])
print("\nTérminos con mayor DF (más comunes entre obras):")
display(estadisticas.sort_values(["df", "termino"], ascending=[False, True]).head(12))
print("\nMuestra de términos poco frecuentes (DF=1):")
display(estadisticas.query("df == 1").sort_values("termino").head(12))


Corpus: 18 documentos | vocabulario: 42,099 términos
Tokens totales: 2,199,219 | longitud promedio: 122,178.83

Tabla pedida (texto y tokens íntegros se conservan en 'documentos'):


,doc_id,titulo,texto,tokens,longitud
0,austen-emma.txt,Emma,"[Emma by Jane Austen 1816]\n\nVOLUME I\n\nCHAPTER I\n\n\nEmma Woodhouse, handsome, cle...","[emma, by, jane, austen, 1816, volume, i, chapter, i, emma, woodhouse, handsome]",161983
1,austen-persuasion.txt,Persuasion,"[Persuasion by Jane Austen 1818]\n\n\nChapter 1\n\n\nSir Walter Elliot, of Kellynch Ha...","[persuasion, by, jane, austen, 1818, chapter, 1, sir, walter, elliot, of, kellynch]",84167
2,austen-sense.txt,Sense and Sensibility,[Sense and Sensibility by Jane Austen 1811]\n\nCHAPTER 1\n\n\nThe family of Dashwood h...,"[sense, and, sensibility, by, jane, austen, 1811, chapter, 1, the, family, of]",120787
3,bible-kjv.txt,King James Bible,[The King James Bible]\n\nThe Old Testament of the King James Bible\n\nThe First Book ...,"[the, king, james, bible, the, old, testament, of, the, king, james, bible]",854046
4,blake-poems.txt,Poems (William Blake),[Poems by William Blake 1789]\n\n \nSONGS OF INNOCENCE AND OF EXPERIENCE\nand THE BOOK...,"[poems, by, william, blake, 1789, songs, of, innocence, and, of, experience, and]",6936
5,bryant-stories.txt,Stories to Tell to Children,[Stories to Tell to Children by Sara Cone Bryant 1918] \r\n\r\n\r\nTWO LITTLE RIDDLES ...,"[stories, to, tell, to, children, by, sara, cone, bryant, 1918, two, little]",46703
6,burgess-busterbrown.txt,The Adventures of Buster Bear,[The Adventures of Buster Bear by Thornton W. Burgess 1920]\r\n\r\nI\r\n\r\nBUSTER BEA...,"[the, adventures, of, buster, bear, by, thornton, w, burgess, 1920, i, buster]",16363
7,carroll-alice.txt,Alice's Adventures in Wonderland,[Alice's Adventures in Wonderland by Lewis Carroll 1865]\n\nCHAPTER I. Down the Rabbit...,"[alice, s, adventures, in, wonderland, by, lewis, carroll, 1865, chapter, i, down]",27336
8,chesterton-ball.txt,The Ball and the Cross,[The Ball and The Cross by G.K. Chesterton 1909]\n\n\nI. A DISCUSSION SOMEWHAT IN THE ...,"[the, ball, and, the, cross, by, g, k, chesterton, 1909, i, a]",82867
9,chesterton-brown.txt,The Wisdom of Father Brown,[The Wisdom of Father Brown by G. K. Chesterton 1914]\n\n\nI. The Absence of Mr Glass\...,"[the, wisdom, of, father, brown, by, g, k, chesterton, 1914, i, the]",73288



Términos con mayor DF (más comunes entre obras):


,termino,df,idf_tfidf,idf_bm25
565,a,18,1.0,0.026668
716,about,18,1.0,0.026668
1326,after,18,1.0,0.026668
1339,against,18,1.0,0.026668
1347,age,18,1.0,0.026668
1612,all,18,1.0,0.026668
1683,alone,18,1.0,0.026668
1736,am,18,1.0,0.026668
1845,among,18,1.0,0.026668
1891,an,18,1.0,0.026668



Muestra de términos poco frecuentes (DF=1):


,termino,df,idf_tfidf,idf_bm25
0,0,1,3.890372,2.538974
1,00,1,3.890372,2.538974
3,00021053,1,3.890372,2.538974
4,00081429,1,3.890372,2.538974
5,00482129,1,3.890372,2.538974
6,01,1,3.890372,2.538974
7,02,1,3.890372,2.538974
11,1000,1,3.890372,2.538974
12,10000,1,3.890372,2.538974
39,1240,1,3.890372,2.538974


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:
# 2. Ocho consultas, DF observado, TF-IDF (disperso) y BM25
consultas = [
    {"id": "C1", "consulta": "the and of", "intencion": "Encontrar obras con conectores extremadamente comunes."},
    {"id": "C2", "consulta": "alice", "intencion": "Encontrar referencias a Alice."},
    {"id": "C3", "consulta": "white rabbit wonderland", "intencion": "Buscar el conejo blanco y el entorno de Wonderland."},
    {"id": "C4", "consulta": "whale harpoon captain ship", "intencion": "Buscar narraciones sobre ballenas y navegación."},
    {"id": "C5", "consulta": "emma marriage", "intencion": "Buscar la historia de Emma y temas de matrimonio."},
    {"id": "C6", "consulta": "king queen ghost revenge", "intencion": "Buscar tragedias con monarquía, fantasmas y venganza."},
    {"id": "C7", "consulta": "god heaven angels satan paradise", "intencion": "Buscar textos sobre teología y el Paraíso."},
    {"id": "C8", "consulta": "grass leaves poetry", "intencion": "Buscar obras líricas con imágenes de hojas y hierba."},
]

# El orden de los ID se trata siempre como cadena.
ids_docs = documentos["doc_id"].astype(str).tolist()
longitudes = documentos["longitud"].to_numpy(dtype=float)


def puntuaciones_tfidf(consulta: str) -> np.ndarray:
    q_conteos = vectorizador.transform([consulta])
    if q_conteos.nnz == 0:
        return np.zeros(N)
    q_tfidf = transformador_tfidf.transform(q_conteos)
    return (X_tfidf @ q_tfidf.T).toarray().ravel()  # coseno por norma L2


def puntuaciones_bm25(consulta: str, k1: float = 1.2, b: float = 0.75) -> np.ndarray:
    if k1 <= 0 or not 0 <= b <= 1:
        raise ValueError("Se requiere k1 > 0 y 0 <= b <= 1")
    q_conteos = vectorizador.transform([consulta])
    puntuaciones = np.zeros(N, dtype=float)
    if q_conteos.nnz == 0:
        return puntuaciones

    # K(d) depende de la longitud, no de la consulta.
    K = k1 * (1.0 - b + b * longitudes / promedio_longitud)
    for col in q_conteos.indices:  # una contribución por término distinto
        inicio, fin = X_csc.indptr[col], X_csc.indptr[col + 1]
        docs_con_termino = X_csc.indices[inicio:fin]
        frecuencias = X_csc.data[inicio:fin]
        aportes = idf_bm25[col] * (
            frecuencias * (k1 + 1.0) / (frecuencias + K[docs_con_termino])
        )
        puntuaciones[docs_con_termino] += aportes
    return puntuaciones


def ranking(puntuaciones: np.ndarray, limite=None) -> list[int]:
    indices_positivos = [i for i, s in enumerate(puntuaciones) if s > 0.0]
    ordenados = sorted(
        indices_positivos, key=lambda i: (-float(puntuaciones[i]), str(ids_docs[i]))
    )
    return ordenados if limite is None else ordenados[:limite]


resultados = {}
info_consultas = []
info_df = []
for consulta in consultas:
    clave, q = consulta["id"], consulta["consulta"]
    indices_q = vectorizador.transform([q]).indices
    palabras_q = tokenizar(q)
    info_consultas.append({
        "ID": clave,
        "Consulta": q,
        "Términos": len(palabras_q),
        "Intención del usuario": consulta["intencion"],
    })
    for t in palabras_q:
        j = indice_vocab.get(t)
        info_df.append({
            "Consulta": clave,
            "Término": t,
            "DF": int(DF[j]) if j is not None else 0,
            "DF / N": round(DF[j] / N, 3) if j is not None else 0.0,
            "IDF BM25": round(float(idf_bm25[j]), 4) if j is not None else np.nan,
            "Tipo (según DF)": (
                "Fuera del vocabulario" if j is None else
                "Muy frecuente" if DF[j] >= np.ceil(0.75 * N) else
                "Poco frecuente" if DF[j] <= max(2, np.floor(0.20 * N)) else
                "Intermedio"
            ),
        })

    s_tf = puntuaciones_tfidf(q)
    s_bm = puntuaciones_bm25(q, k1=1.2, b=0.75)
    resultados[clave] = {
        "consulta": q,
        "q_indices": indices_q,
        "scores_tfidf": s_tf,
        "scores_bm25": s_bm,
        "full_tfidf": ranking(s_tf),
        "full_bm25": ranking(s_bm),
        "top_tfidf": ranking(s_tf, limite=10),
        "top_bm25": ranking(s_bm, limite=10),
    }

print("Consultas e intención de búsqueda:")
display(pd.DataFrame(info_consultas))
print("\nJustificación con DF empírico (N =", N, "):")
display(pd.DataFrame(info_df))
print("\nCobertura de los rankings (resultados con score > 0):")
display(pd.DataFrame([
    {
        "Consulta": c["id"],
        "TF-IDF top": len(resultados[c["id"]]["top_tfidf"]),
        "BM25 top": len(resultados[c["id"]]["top_bm25"]),
    }
    for c in consultas
]))

# Caso límite: no se inventan resultados si no hay palabras del vocabulario.
consulta_sin_vocabulario = "zzzxqv_no_aparece_987654"
assert vectorizador.transform([consulta_sin_vocabulario]).nnz == 0
assert ranking(puntuaciones_tfidf(consulta_sin_vocabulario), 10) == []
assert ranking(puntuaciones_bm25(consulta_sin_vocabulario), 10) == []
print("\nPrueba de consulta sin vocabulario: ambos rankings son [].")


Consultas e intención de búsqueda:


,ID,Consulta,Términos,Intención del usuario
0,C1,the and of,3,Encontrar obras con conectores extremadamente comunes.
1,C2,alice,1,Encontrar referencias a Alice.
2,C3,white rabbit wonderland,3,Buscar el conejo blanco y el entorno de Wonderland.
3,C4,whale harpoon captain ship,4,Buscar narraciones sobre ballenas y navegación.
4,C5,emma marriage,2,Buscar la historia de Emma y temas de matrimonio.
5,C6,king queen ghost revenge,4,"Buscar tragedias con monarquía, fantasmas y venganza."
6,C7,god heaven angels satan paradise,5,Buscar textos sobre teología y el Paraíso.
7,C8,grass leaves poetry,3,Buscar obras líricas con imágenes de hojas y hierba.



Justificación con DF empírico (N = 18 ):


,Consulta,Término,DF,DF / N,IDF BM25,Tipo (según DF)
0,C1,the,18,1.000,0.0267,Muy frecuente
1,C1,and,18,1.000,0.0267,Muy frecuente
2,C1,of,18,1.000,0.0267,Muy frecuente
3,C2,alice,3,0.167,1.6917,Poco frecuente
4,C3,white,16,0.889,0.1411,Muy frecuente
5,C3,rabbit,6,0.333,1.0726,Intermedio
6,C3,wonderland,2,0.111,2.0281,Poco frecuente
7,C4,whale,6,0.333,1.0726,Intermedio
8,C4,harpoon,2,0.111,2.0281,Poco frecuente
9,C4,captain,9,0.500,0.6931,Intermedio



Cobertura de los rankings (resultados con score > 0):


,Consulta,TF-IDF top,BM25 top
0,C1,10,10
1,C2,3,3
2,C3,10,10
3,C4,10,10
4,C5,10,10
5,C6,10,10
6,C7,10,10
7,C8,10,10



Prueba de consulta sin vocabulario: ambos rankings son [].


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [3]:
# 3. Tablas comparativas de rankings y análisis término a término
from itertools import combinations


def etiqueta_doc(i: int) -> str:
    return f"{ids_docs[i]} — {documentos.at[i, 'titulo']}"


resumen_rankings = []
for c in consultas:
    clave = c["id"]
    r = resultados[clave]
    lista_tf, lista_bm = r["top_tfidf"], r["top_bm25"]
    solapamiento = len(set(ids_docs[i] for i in lista_tf) & set(ids_docs[i] for i in lista_bm))
    resumen_rankings.append({
        "Consulta": clave,
        "Tamaño TF-IDF": len(lista_tf),
        "Tamaño BM25": len(lista_bm),
        "IDs compartidos": solapamiento,
        "Primero TF-IDF": ids_docs[lista_tf[0]] if lista_tf else "—",
        "Primero BM25": ids_docs[lista_bm[0]] if lista_bm else "—",
    })
    print(f"\n{clave} — {r['consulta']!r} | top10 compartidos: {solapamiento}")
    filas = []
    for pos in range(max(len(lista_tf), len(lista_bm))):
        a = lista_tf[pos] if pos < len(lista_tf) else None
        b = lista_bm[pos] if pos < len(lista_bm) else None
        filas.append({
            "Posición": pos + 1,
            "ID y título TF-IDF": etiqueta_doc(a) if a is not None else "—",
            "Coseno": round(float(r["scores_tfidf"][a]), 6) if a is not None else np.nan,
            "ID y título BM25": etiqueta_doc(b) if b is not None else "—",
            "Score BM25": round(float(r["scores_bm25"][b]), 6) if b is not None else np.nan,
        })
    display(pd.DataFrame(filas, columns=[
        "Posición", "ID y título TF-IDF", "Coseno", "ID y título BM25", "Score BM25"
    ]))

print("\nResumen de comparación por consulta:")
display(pd.DataFrame(resumen_rankings))

# Prioriza inversiones reales de orden; luego cambios de distancia entre posiciones.
candidatos = []
for c in consultas:
    clave = c["id"]
    r = resultados[clave]
    comunes = sorted(set(r["top_tfidf"]) & set(r["top_bm25"]))
    pos_tf = {idx: pos + 1 for pos, idx in enumerate(r["full_tfidf"])}
    pos_bm = {idx: pos + 1 for pos, idx in enumerate(r["full_bm25"])}
    for a, b_ in combinations(comunes, 2):
        diff_tf = pos_tf[a] - pos_tf[b_]
        diff_bm = pos_bm[a] - pos_bm[b_]
        inversion = diff_tf * diff_bm < 0
        cambio_distancia = abs(diff_tf - diff_bm)
        if cambio_distancia > 0:
            candidatos.append((int(inversion), cambio_distancia, clave, a, b_))

if not candidatos:
    raise RuntimeError("No se encontró un par para examinar diferencias de ranking.")
_, _, clave_elegida, doc_a, doc_b = max(candidatos)
r_elegido = resultados[clave_elegida]
consulta_elegida = r_elegido["consulta"]


def detalle_aportes(consulta: str, i: int, k1=1.2, b=0.75) -> list[dict]:
    """Aporte exacto por término de la consulta para un documento dado."""
    j_terminos = vectorizador.transform([consulta]).indices
    K_i = k1 * (1 - b + b * longitudes[i] / promedio_longitud)
    filas = []
    for j in j_terminos:
        f = float(X_conteos[i, j])
        aporte = float(idf_bm25[j] * f * (k1 + 1) / (f + K_i)) if f else 0.0
        filas.append({
            "Documento": ids_docs[i],
            "|d|": int(longitudes[i]),
            "Término": str(vocabulario[j]),
            "f(t,d)": int(f),
            "df(t)": int(DF[j]),
            "IDF BM25": float(idf_bm25[j]),
            "Aporte BM25": aporte,
        })
    return filas

pos_tf_e = {idx: p + 1 for p, idx in enumerate(r_elegido["full_tfidf"])}
pos_bm_e = {idx: p + 1 for p, idx in enumerate(r_elegido["full_bm25"])}
print(f"\nANÁLISIS DE CAMBIO: {clave_elegida} — {consulta_elegida!r}")
comparacion_dos = []
for idx in [doc_a, doc_b]:
    comparacion_dos.append({
        "Documento": etiqueta_doc(idx),
        "|d|": int(longitudes[idx]),
        "Posición coseno": pos_tf_e[idx],
        "Score coseno": r_elegido["scores_tfidf"][idx],
        "Posición BM25": pos_bm_e[idx],
        "Score BM25": r_elegido["scores_bm25"][idx],
    })
display(pd.DataFrame(comparacion_dos).round(6))

aportes_elegidos = pd.DataFrame(
    detalle_aportes(consulta_elegida, doc_a) + detalle_aportes(consulta_elegida, doc_b)
)
display(aportes_elegidos.round(6))

# Comprobación: la suma por término reproduce el score BM25.
for idx in [doc_a, doc_b]:
    suma = aportes_elegidos.loc[aportes_elegidos["Documento"] == ids_docs[idx], "Aporte BM25"].sum()
    assert np.isclose(suma, r_elegido["scores_bm25"][idx], rtol=1e-10)

orden_coseno = [doc_a, doc_b] if pos_tf_e[doc_a] < pos_tf_e[doc_b] else [doc_b, doc_a]
orden_bm25 = [doc_a, doc_b] if pos_bm_e[doc_a] < pos_bm_e[doc_b] else [doc_b, doc_a]
print("Interpretación del cambio:")
print(f"- TF-IDF + coseno prioriza {ids_docs[orden_coseno[0]]} frente a {ids_docs[orden_coseno[1]]} "
      f"por scores {r_elegido['scores_tfidf'][orden_coseno[0]]:.6f} > "
      f"{r_elegido['scores_tfidf'][orden_coseno[1]]:.6f}.")
print(f"- BM25 prioriza {ids_docs[orden_bm25[0]]} frente a {ids_docs[orden_bm25[1]]} "
      f"por scores {r_elegido['scores_bm25'][orden_bm25[0]]:.6f} > "
      f"{r_elegido['scores_bm25'][orden_bm25[1]]:.6f}.")
print(f"- Longitudes: {ids_docs[doc_a]} tiene {int(longitudes[doc_a]):,} tokens, "
      f"{ids_docs[doc_b]} tiene {int(longitudes[doc_b]):,}; "
      f"promedio de colección = {promedio_longitud:,.1f}.")

# Resalta el término cuya aportación es más diferente entre los dos documentos.
tabla_pivote = aportes_elegidos.pivot(index="Término", columns="Documento", values="Aporte BM25").fillna(0)
tabla_pivote["Diferencia absoluta"] = (tabla_pivote[ids_docs[doc_a]] - tabla_pivote[ids_docs[doc_b]]).abs()
termino_destacado = str(tabla_pivote["Diferencia absoluta"].idxmax())
j_dest = indice_vocab[termino_destacado]
print(f"- El término con mayor diferencia de aporte es {termino_destacado!r}: "
      f"DF={DF[j_dest]}/{N}, IDF BM25={idf_bm25[j_dest]:.4f}, "
      f"frecuencias=({int(X_conteos[doc_a,j_dest])}, {int(X_conteos[doc_b,j_dest])}).")
print("- Razón del resultado: BM25 suma IDF positivo (rareza), frecuencia con "
      "saturación (k1) y ajuste por longitud (b); el coseno normaliza vectores "
      "TF-IDF completos. La tabla permite identificar qué factor favorece a cada obra.")



C1 — 'the and of' | top10 compartidos: 7


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,bible-kjv.txt — King James Bible,0.822548,bible-kjv.txt — King James Bible,0.175986
1,2,whitman-leaves.txt — Leaves of Grass,0.814101,whitman-leaves.txt — Leaves of Grass,0.175973
2,3,blake-poems.txt — Poems (William Blake),0.750558,melville-moby_dick.txt — Moby Dick,0.175968
3,4,melville-moby_dick.txt — Moby Dick,0.736252,chesterton-ball.txt — The Ball and the Cross,0.175959
4,5,milton-paradise.txt — Paradise Lost,0.718224,edgeworth-parents.txt — The Parent's Assistant,0.175956
5,6,bryant-stories.txt — Stories to Tell to Children,0.709688,austen-emma.txt — Emma,0.175955
6,7,chesterton-brown.txt — The Wisdom of Father Brown,0.703756,chesterton-brown.txt — The Wisdom of Father Brown,0.175954
7,8,chesterton-ball.txt — The Ball and the Cross,0.688887,austen-persuasion.txt — Persuasion,0.175954
8,9,chesterton-thursday.txt — The Man Who Was Thursday,0.655344,austen-sense.txt — Sense and Sensibility,0.175954
9,10,austen-persuasion.txt — Persuasion,0.622913,milton-paradise.txt — Paradise Lost,0.175952



C2 — 'alice' | top10 compartidos: 3


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,carroll-alice.txt — Alice's Adventures in Wonderland,0.370479,carroll-alice.txt — Alice's Adventures in Wonderland,3.717005
1,2,chesterton-thursday.txt — The Man Who Was Thursday,0.000451,chesterton-thursday.txt — The Man Who Was Thursday,2.148020
2,3,edgeworth-parents.txt — The Parent's Assistant,0.000358,edgeworth-parents.txt — The Parent's Assistant,2.091937



C3 — 'white rabbit wonderland' | top10 compartidos: 10


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,carroll-alice.txt — Alice's Adventures in Wonderland,0.024499,carroll-alice.txt — Alice's Adventures in Wonderland,6.465118
1,2,burgess-busterbrown.txt — The Adventures of Buster Bear,0.006604,chesterton-thursday.txt — The Man Who Was Thursday,4.243467
2,3,blake-poems.txt — Poems (William Blake),0.005243,bryant-stories.txt — Stories to Tell to Children,2.523013
3,4,bryant-stories.txt — Stories to Tell to Children,0.004858,burgess-busterbrown.txt — The Adventures of Buster Bear,2.272907
4,5,melville-moby_dick.txt — Moby Dick,0.004091,chesterton-ball.txt — The Ball and the Cross,2.228089
5,6,chesterton-thursday.txt — The Man Who Was Thursday,0.003332,chesterton-brown.txt — The Wisdom of Father Brown,2.149368
6,7,chesterton-brown.txt — The Wisdom of Father Brown,0.002790,melville-moby_dick.txt — Moby Dick,0.308277
7,8,chesterton-ball.txt — The Ball and the Cross,0.002758,whitman-leaves.txt — Leaves of Grass,0.306133
8,9,whitman-leaves.txt — Leaves of Grass,0.001996,edgeworth-parents.txt — The Parent's Assistant,0.302643
9,10,edgeworth-parents.txt — The Parent's Assistant,0.001227,blake-poems.txt — Poems (William Blake),0.301550


... (12 outputs left out)


ANÁLISIS DE CAMBIO: C6 — 'king queen ghost revenge'


,Documento,|d|,Posición coseno,Score coseno,Posición BM25,Score BM25
0,melville-moby_dick.txt — Moby Dick,218621,9,0.002765,1,3.597908
1,shakespeare-hamlet.txt — Hamlet,30271,1,0.037767,5,2.354863


,Documento,|d|,Término,"f(t,d)",df(t),IDF BM25,Aporte BM25
0,melville-moby_dick.txt,218621,ghost,13,9,0.693147,1.329541
1,melville-moby_dick.txt,218621,king,64,15,0.203599,0.434935
2,melville-moby_dick.txt,218621,queen,18,15,0.203599,0.404940
3,melville-moby_dick.txt,218621,revenge,8,8,0.804373,1.428493
4,shakespeare-hamlet.txt,30271,ghost,21,9,0.693147,1.487870
5,shakespeare-hamlet.txt,30271,king,172,15,0.203599,0.446560
6,shakespeare-hamlet.txt,30271,queen,8,15,0.203599,0.420433
7,shakespeare-hamlet.txt,30271,revenge,0,8,0.804373,0.000000


Interpretación del cambio:
- TF-IDF + coseno prioriza shakespeare-hamlet.txt frente a melville-moby_dick.txt por scores 0.037767 > 0.002765.
- BM25 prioriza melville-moby_dick.txt frente a shakespeare-hamlet.txt por scores 3.597908 > 2.354863.
- Longitudes: melville-moby_dick.txt tiene 218,621 tokens, shakespeare-hamlet.txt tiene 30,271; promedio de colección = 122,178.8.
- El término con mayor diferencia de aporte es 'revenge': DF=8/18, IDF BM25=0.8044, frecuencias=(8, 0).
- Razón del resultado: BM25 suma IDF positivo (rareza), frecuencia con saturación (k1) y ajuste por longitud (b); el coseno normaliza vectores TF-IDF completos. La tabla permite identificar qué factor favorece a cada obra.


**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [4]:
# 3. Tablas comparativas de rankings y análisis término a término
from itertools import combinations


def etiqueta_doc(i: int) -> str:
    return f"{ids_docs[i]} — {documentos.at[i, 'titulo']}"


resumen_rankings = []
for c in consultas:
    clave = c["id"]
    r = resultados[clave]
    lista_tf, lista_bm = r["top_tfidf"], r["top_bm25"]
    solapamiento = len(set(ids_docs[i] for i in lista_tf) & set(ids_docs[i] for i in lista_bm))
    resumen_rankings.append({
        "Consulta": clave,
        "Tamaño TF-IDF": len(lista_tf),
        "Tamaño BM25": len(lista_bm),
        "IDs compartidos": solapamiento,
        "Primero TF-IDF": ids_docs[lista_tf[0]] if lista_tf else "—",
        "Primero BM25": ids_docs[lista_bm[0]] if lista_bm else "—",
    })
    print(f"\n{clave} — {r['consulta']!r} | top10 compartidos: {solapamiento}")
    filas = []
    for pos in range(max(len(lista_tf), len(lista_bm))):
        a = lista_tf[pos] if pos < len(lista_tf) else None
        b = lista_bm[pos] if pos < len(lista_bm) else None
        filas.append({
            "Posición": pos + 1,
            "ID y título TF-IDF": etiqueta_doc(a) if a is not None else "—",
            "Coseno": round(float(r["scores_tfidf"][a]), 6) if a is not None else np.nan,
            "ID y título BM25": etiqueta_doc(b) if b is not None else "—",
            "Score BM25": round(float(r["scores_bm25"][b]), 6) if b is not None else np.nan,
        })
    display(pd.DataFrame(filas, columns=[
        "Posición", "ID y título TF-IDF", "Coseno", "ID y título BM25", "Score BM25"
    ]))

print("\nResumen de comparación por consulta:")
display(pd.DataFrame(resumen_rankings))

# Prioriza inversiones reales de orden; luego cambios de distancia entre posiciones.
candidatos = []
for c in consultas:
    clave = c["id"]
    r = resultados[clave]
    comunes = sorted(set(r["top_tfidf"]) & set(r["top_bm25"]))
    pos_tf = {idx: pos + 1 for pos, idx in enumerate(r["full_tfidf"])}
    pos_bm = {idx: pos + 1 for pos, idx in enumerate(r["full_bm25"])}
    for a, b_ in combinations(comunes, 2):
        diff_tf = pos_tf[a] - pos_tf[b_]
        diff_bm = pos_bm[a] - pos_bm[b_]
        inversion = diff_tf * diff_bm < 0
        cambio_distancia = abs(diff_tf - diff_bm)
        if cambio_distancia > 0:
            candidatos.append((int(inversion), cambio_distancia, clave, a, b_))

if not candidatos:
    raise RuntimeError("No se encontró un par para examinar diferencias de ranking.")
_, _, clave_elegida, doc_a, doc_b = max(candidatos)
r_elegido = resultados[clave_elegida]
consulta_elegida = r_elegido["consulta"]


def detalle_aportes(consulta: str, i: int, k1=1.2, b=0.75) -> list[dict]:
    """Aporte exacto por término de la consulta para un documento dado."""
    j_terminos = vectorizador.transform([consulta]).indices
    K_i = k1 * (1 - b + b * longitudes[i] / promedio_longitud)
    filas = []
    for j in j_terminos:
        f = float(X_conteos[i, j])
        aporte = float(idf_bm25[j] * f * (k1 + 1) / (f + K_i)) if f else 0.0
        filas.append({
            "Documento": ids_docs[i],
            "|d|": int(longitudes[i]),
            "Término": str(vocabulario[j]),
            "f(t,d)": int(f),
            "df(t)": int(DF[j]),
            "IDF BM25": float(idf_bm25[j]),
            "Aporte BM25": aporte,
        })
    return filas

pos_tf_e = {idx: p + 1 for p, idx in enumerate(r_elegido["full_tfidf"])}
pos_bm_e = {idx: p + 1 for p, idx in enumerate(r_elegido["full_bm25"])}
print(f"\nANÁLISIS DE CAMBIO: {clave_elegida} — {consulta_elegida!r}")
comparacion_dos = []
for idx in [doc_a, doc_b]:
    comparacion_dos.append({
        "Documento": etiqueta_doc(idx),
        "|d|": int(longitudes[idx]),
        "Posición coseno": pos_tf_e[idx],
        "Score coseno": r_elegido["scores_tfidf"][idx],
        "Posición BM25": pos_bm_e[idx],
        "Score BM25": r_elegido["scores_bm25"][idx],
    })
display(pd.DataFrame(comparacion_dos).round(6))

aportes_elegidos = pd.DataFrame(
    detalle_aportes(consulta_elegida, doc_a) + detalle_aportes(consulta_elegida, doc_b)
)
display(aportes_elegidos.round(6))

# Comprobación: la suma por término reproduce el score BM25.
for idx in [doc_a, doc_b]:
    suma = aportes_elegidos.loc[aportes_elegidos["Documento"] == ids_docs[idx], "Aporte BM25"].sum()
    assert np.isclose(suma, r_elegido["scores_bm25"][idx], rtol=1e-10)

orden_coseno = [doc_a, doc_b] if pos_tf_e[doc_a] < pos_tf_e[doc_b] else [doc_b, doc_a]
orden_bm25 = [doc_a, doc_b] if pos_bm_e[doc_a] < pos_bm_e[doc_b] else [doc_b, doc_a]
print("Interpretación del cambio:")
print(f"- TF-IDF + coseno prioriza {ids_docs[orden_coseno[0]]} frente a {ids_docs[orden_coseno[1]]} "
      f"por scores {r_elegido['scores_tfidf'][orden_coseno[0]]:.6f} > "
      f"{r_elegido['scores_tfidf'][orden_coseno[1]]:.6f}.")
print(f"- BM25 prioriza {ids_docs[orden_bm25[0]]} frente a {ids_docs[orden_bm25[1]]} "
      f"por scores {r_elegido['scores_bm25'][orden_bm25[0]]:.6f} > "
      f"{r_elegido['scores_bm25'][orden_bm25[1]]:.6f}.")
print(f"- Longitudes: {ids_docs[doc_a]} tiene {int(longitudes[doc_a]):,} tokens, "
      f"{ids_docs[doc_b]} tiene {int(longitudes[doc_b]):,}; "
      f"promedio de colección = {promedio_longitud:,.1f}.")

# Resalta el término cuya aportación es más diferente entre los dos documentos.
tabla_pivote = aportes_elegidos.pivot(index="Término", columns="Documento", values="Aporte BM25").fillna(0)
tabla_pivote["Diferencia absoluta"] = (tabla_pivote[ids_docs[doc_a]] - tabla_pivote[ids_docs[doc_b]]).abs()
termino_destacado = str(tabla_pivote["Diferencia absoluta"].idxmax())
j_dest = indice_vocab[termino_destacado]
print(f"- El término con mayor diferencia de aporte es {termino_destacado!r}: "
      f"DF={DF[j_dest]}/{N}, IDF BM25={idf_bm25[j_dest]:.4f}, "
      f"frecuencias=({int(X_conteos[doc_a,j_dest])}, {int(X_conteos[doc_b,j_dest])}).")
print("- Razón del resultado: BM25 suma IDF positivo (rareza), frecuencia con "
      "saturación (k1) y ajuste por longitud (b); el coseno normaliza vectores "
      "TF-IDF completos. La tabla permite identificar qué factor favorece a cada obra.")



C1 — 'the and of' | top10 compartidos: 7


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,bible-kjv.txt — King James Bible,0.822548,bible-kjv.txt — King James Bible,0.175986
1,2,whitman-leaves.txt — Leaves of Grass,0.814101,whitman-leaves.txt — Leaves of Grass,0.175973
2,3,blake-poems.txt — Poems (William Blake),0.750558,melville-moby_dick.txt — Moby Dick,0.175968
3,4,melville-moby_dick.txt — Moby Dick,0.736252,chesterton-ball.txt — The Ball and the Cross,0.175959
4,5,milton-paradise.txt — Paradise Lost,0.718224,edgeworth-parents.txt — The Parent's Assistant,0.175956
5,6,bryant-stories.txt — Stories to Tell to Children,0.709688,austen-emma.txt — Emma,0.175955
6,7,chesterton-brown.txt — The Wisdom of Father Brown,0.703756,chesterton-brown.txt — The Wisdom of Father Brown,0.175954
7,8,chesterton-ball.txt — The Ball and the Cross,0.688887,austen-persuasion.txt — Persuasion,0.175954
8,9,chesterton-thursday.txt — The Man Who Was Thursday,0.655344,austen-sense.txt — Sense and Sensibility,0.175954
9,10,austen-persuasion.txt — Persuasion,0.622913,milton-paradise.txt — Paradise Lost,0.175952



C2 — 'alice' | top10 compartidos: 3


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,carroll-alice.txt — Alice's Adventures in Wonderland,0.370479,carroll-alice.txt — Alice's Adventures in Wonderland,3.717005
1,2,chesterton-thursday.txt — The Man Who Was Thursday,0.000451,chesterton-thursday.txt — The Man Who Was Thursday,2.148020
2,3,edgeworth-parents.txt — The Parent's Assistant,0.000358,edgeworth-parents.txt — The Parent's Assistant,2.091937



C3 — 'white rabbit wonderland' | top10 compartidos: 10


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,carroll-alice.txt — Alice's Adventures in Wonderland,0.024499,carroll-alice.txt — Alice's Adventures in Wonderland,6.465118
1,2,burgess-busterbrown.txt — The Adventures of Buster Bear,0.006604,chesterton-thursday.txt — The Man Who Was Thursday,4.243467
2,3,blake-poems.txt — Poems (William Blake),0.005243,bryant-stories.txt — Stories to Tell to Children,2.523013
3,4,bryant-stories.txt — Stories to Tell to Children,0.004858,burgess-busterbrown.txt — The Adventures of Buster Bear,2.272907
4,5,melville-moby_dick.txt — Moby Dick,0.004091,chesterton-ball.txt — The Ball and the Cross,2.228089
5,6,chesterton-thursday.txt — The Man Who Was Thursday,0.003332,chesterton-brown.txt — The Wisdom of Father Brown,2.149368
6,7,chesterton-brown.txt — The Wisdom of Father Brown,0.002790,melville-moby_dick.txt — Moby Dick,0.308277
7,8,chesterton-ball.txt — The Ball and the Cross,0.002758,whitman-leaves.txt — Leaves of Grass,0.306133
8,9,whitman-leaves.txt — Leaves of Grass,0.001996,edgeworth-parents.txt — The Parent's Assistant,0.302643
9,10,edgeworth-parents.txt — The Parent's Assistant,0.001227,blake-poems.txt — Poems (William Blake),0.301550


... (12 outputs left out)


ANÁLISIS DE CAMBIO: C6 — 'king queen ghost revenge'


,Documento,|d|,Posición coseno,Score coseno,Posición BM25,Score BM25
0,melville-moby_dick.txt — Moby Dick,218621,9,0.002765,1,3.597908
1,shakespeare-hamlet.txt — Hamlet,30271,1,0.037767,5,2.354863


,Documento,|d|,Término,"f(t,d)",df(t),IDF BM25,Aporte BM25
0,melville-moby_dick.txt,218621,ghost,13,9,0.693147,1.329541
1,melville-moby_dick.txt,218621,king,64,15,0.203599,0.434935
2,melville-moby_dick.txt,218621,queen,18,15,0.203599,0.404940
3,melville-moby_dick.txt,218621,revenge,8,8,0.804373,1.428493
4,shakespeare-hamlet.txt,30271,ghost,21,9,0.693147,1.487870
5,shakespeare-hamlet.txt,30271,king,172,15,0.203599,0.446560
6,shakespeare-hamlet.txt,30271,queen,8,15,0.203599,0.420433
7,shakespeare-hamlet.txt,30271,revenge,0,8,0.804373,0.000000


Interpretación del cambio:
- TF-IDF + coseno prioriza shakespeare-hamlet.txt frente a melville-moby_dick.txt por scores 0.037767 > 0.002765.
- BM25 prioriza melville-moby_dick.txt frente a shakespeare-hamlet.txt por scores 3.597908 > 2.354863.
- Longitudes: melville-moby_dick.txt tiene 218,621 tokens, shakespeare-hamlet.txt tiene 30,271; promedio de colección = 122,178.8.
- El término con mayor diferencia de aporte es 'revenge': DF=8/18, IDF BM25=0.8044, frecuencias=(8, 0).
- Razón del resultado: BM25 suma IDF positivo (rareza), frecuencia con saturación (k1) y ajuste por longitud (b); el coseno normaliza vectores TF-IDF completos. La tabla permite identificar qué factor favorece a cada obra.
